# 06 · Demo: baseline vs. condition-aware search (Gradio)

**Purpose.** An interactive interface for the live demo and the user study: one query box, skill and day selectors, and the **baseline and improved rankings side by side**, with explanations, highlighted matching terms, per-query latency, and a switch between **live data** and a **saved snapshot**.

**Inputs:** the search functions from `04_search` and the live-collection function from `00_data_collection` (both loaded with `%run`), snapshots in `data/snapshots/`.

**Outputs:** a local web app (URL printed below). Open it in a browser and press F11 for full-screen presentation.

**Presentation tips**
- Rehearse with *Saved snapshot* on the snapshot you will present. If the network or the APIs fail on the day, the demo is identical.
- *Live data* falls back to the latest snapshot automatically (with a visible notice) when an API is down.
- For the user study, hide the side-by-side view and show one system at a time, labelled neutrally (see `SHOW_SYSTEM_NAMES`).

In [17]:
# ---- Configuration ----
import os
from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file():
            return folder
    raise FileNotFoundError("Project root not found. Open the project folder in VS Code (File > Open Folder).")


ROOT = find_project_root()
os.chdir(ROOT / "notebooks")
os.environ["SURF_IR_LIBRARY_MODE"] = "1"
%run ./04_search.ipynb
%run ./00_data_collection.ipynb
# Settings below are applied after %run so they take precedence.

DEMO_SNAPSHOT_DATE = None        # None = latest snapshot; set a date to rehearse/present a fixed day
DEFAULT_K = 5
SERVER_NAME = "127.0.0.1"        # local only; nothing is exposed to the internet
SERVER_PORT = None               # None = first free port from 7860
SHARE = False                    # never create a public Gradio link for the presentation
LIVE_CACHE_MINUTES = 30          # reuse a live fetch for this long (avoids hammering the free APIs)
SHOW_SYSTEM_NAMES = True         # False = label columns "System A" / "System B" (user study)
MAP_FILE = ROOT / "results" / "figures" / "surf_spot_map.html"   # created by 07_spot_map.ipynb
MAP_URL = "/gradio_api/file=" + MAP_FILE.as_posix()               # served by the demo server (see allowed_paths)
SAMPLE_N = None                  # not used here
SEED = 42

Library mode: functions defined, nothing fetched.


## 1. Data source: live or saved snapshot
Live mode calls the same `collect_all()` used for daily snapshots (notebook 00), but keeps the result in memory. If the fetch fails (API down, no internet), we fall back to the latest saved snapshot and say so in the interface. A demo must never show an error page.

In [18]:
import time

import gradio as gr

_LIVE = {"snapshot": None, "fetched": None}


def get_live_snapshot(progress=None):
    """Returns (snapshot, notice). Live data is cached for LIVE_CACHE_MINUTES.
    `progress(done, total, label)` is called before each download (used for the progress bar)."""
    now = pd.Timestamp.now()
    if _LIVE["snapshot"] is not None and now - _LIVE["fetched"] < pd.Timedelta(minutes=LIVE_CACHE_MINUTES):
        return _LIVE["snapshot"], None
    try:
        data = collect_all(progress=progress)
        if data["meta"]["status"] == "failed":
            raise RuntimeError("no marine forecast could be fetched")
        snap = snapshot_from_frames(data["marine"], data["wind"], data["obs"], data["meta"],
                                    key=f"live-{data['meta']['fetched_at']}")
        snap["ref_time"] = pd.Timestamp(data["meta"]["fetched_at"])
        _LIVE.update(snapshot=snap, fetched=now)
        return snap, None
    except Exception as exc:
        fallback = load_snapshot(DEMO_SNAPSHOT_DATE)
        return fallback, f"Live data unavailable ({exc}). Showing saved snapshot {fallback['date']}."


def pick_snapshot(source, snapshot_date, progress=None):
    if source == "Live data":
        return get_live_snapshot(progress)
    return load_snapshot(snapshot_date or DEMO_SNAPSHOT_DATE), None

## 2. Rendering results
Below the results, a **score breakdown** shows for every improved result the conditions at the best hour, each factor (the lowest one, named in *"Limited by …"*, in red), the product that gives CF, and how text score, CF and α add up to the final score. Below that, a **forecast table** (like a surf-forecast app) shows the next 48 hours in 3-hour steps for the top 3 spots of the improved system: temperature, wind and gusts, waves, swell, swell period and wave power, with your query's time window highlighted.

Each result card shows: the spot, the verdict for the user's level, the explanation (improved system only), a safety warning if any, and the profile text with the **matched terms highlighted**. The highlighting is computed from the stems that actually matched, so the audience can see query expansion at work (e.g. *newbie* highlighting *beginner*).

In [19]:
CSS = """
.card {border: 1px solid #d0d7de; border-radius: 8px; padding: 10px 12px; margin-bottom: 8px; background: #fff; color: #1f2328;}
.card h4 {margin: 0 0 4px 0;}
.meta {color: #57606a; font-size: 0.85em;}
.badge {display: inline-block; padding: 1px 8px; border-radius: 10px; font-size: 0.8em; font-weight: 600; margin-left: 6px;}
.Good {background: #dafbe1; color: #116329;} .Fair {background: #fff8c5; color: #7d4e00;}
.Poor {background: #ffebe9; color: #a40e26;} .Nodata {background: #eaeef2; color: #57606a;}
.warn {color: #a40e26; font-weight: 600;}
.expl {margin: 6px 0;}
.profile {font-size: 0.85em; color: #424a53;}
mark {background: #fff3a3; padding: 0 1px;}
.outlook td {padding: 2px 6px; text-align: center; font-size: 0.85em;}
.fc-spot {margin: 14px 0;}
.fc-scroll {overflow-x: auto;}
.fc {border-collapse: collapse; font-size: 0.85em; margin-top: 4px;}
.fc td, .fc th {border: 1px solid #eaeef2; padding: 3px 6px; text-align: center; white-space: nowrap; color: #1f2328;}
.fc th.fc-label {text-align: left; background: #f6f8fa; font-weight: 600;}
.fc .fc-unit {font-weight: 400; color: #57606a; font-size: 0.85em;}
.fc th.fc-day {background: #f6f8fa;}
.fc th.fc-win {background: #fff3a3;}
.bd {border-collapse: collapse; font-size: 0.85em; margin: 4px 0 12px 0;}
.bd td, .bd th {border: 1px solid #eaeef2; padding: 4px 7px; text-align: center; vertical-align: middle; color: #1f2328;}
.bd th {background: #f6f8fa;}
.bd td.bd-left {text-align: left;}
.bd td.bd-limit {background: #ffebe9; color: #a40e26; font-weight: 700;}
.bd-limit-key {background: #ffebe9; color: #a40e26; font-weight: 700; padding: 0 4px;}
.bd-small {font-size: 0.8em; color: #57606a; font-weight: 400;}
"""

DAY_CHOICES = ["From query", "Today", "Tomorrow", "Day after tomorrow"]
SKILL_CHOICES = ["From query", "beginner", "intermediate", "advanced"]


def render_cards(info, show_conditions):
    if not info["results"]:
        return "<p>No results.</p>"
    stems = set(analyze(info["parsed"].text)) | {s for exp in info["expansions"] for s in analyze(exp)}
    html = []
    for r in info["results"]:
        spot = SPOTS[r["spot_id"]]
        badge = f"<span class='badge {r['verdict'].replace(' ', '')}'>{escape(r['verdict'])}</span>" if show_conditions and r["verdict"] else ""
        dist = f" · {r['distance_km']:.0f} km away" if r["distance_km"] is not None else ""
        body = (f"<div class='expl'>{escape(r['explanation'])}</div>" if show_conditions and r["explanation"]
                else "<div class='expl meta'>Text match only (no condition information).</div>")
        warn = f"<div class='warn'>⚠ {escape(r['warning'])}</div>" if show_conditions and r["warning"] else ""
        matched = ", ".join(r["matched_stems"]) or "none"
        map_link = (f" · <a href='{MAP_URL}#{r['spot_id']}' target='_blank'>📍 Show on map</a>" if MAP_FILE.exists() else "")
        html.append(
            f"<div class='card'><h4>{r['rank']}. {escape(r['name'])}{badge}</h4>"
            f"<div class='meta'>{escape(spot['region'])} · {escape(spot['break_type'])} break{dist} · score {r['score']:.2f}{map_link}</div>"
            f"{body}{warn}<div class='profile'>{highlight(spot['description'], stems)}</div>"
            f"<div class='meta'>Matched terms: {escape(matched)}</div></div>")
    return "".join(html)


def render_outlook(results, snap, skill):
    """7-day timeline: best condition-fit score per day for the top 3 improved results."""
    if not results:
        return ""
    rows = {r["name"]: daily_outlook(r["spot_id"], snap, skill) for r in results[:3]}
    days = sorted({d for s in rows.values() for d in s.index})
    head = "".join(f"<th>{pd.Timestamp(d):%a %d}</th>" for d in days)
    body = ""
    for name, s in rows.items():
        cells = ""
        for d in days:
            v = s.get(d, np.nan)
            color = "#eaeef2" if pd.isna(v) else ("#dafbe1" if v >= THRESHOLDS[skill]["good_threshold"]
                                                  else "#fff8c5" if v >= THRESHOLDS[skill]["surfable_threshold"] else "#ffebe9")
            cells += f"<td style='background:{color}'>{'' if pd.isna(v) else f'{v:.2f}'}</td>"
        body += f"<tr><td style='text-align:left'>{escape(name)}</td>{cells}</tr>"
    return (f"<b>7-day outlook ({escape(skill)}, best condition-fit score per day)</b>"
            f"<table class='outlook'><tr><th></th>{head}</tr>{body}</table>")


FORECAST_STEP_HOURS = 3        # one column every 3 hours, like common surf-forecast tables
FORECAST_COLUMNS = 16          # 16 x 3 h = 48 hours


def wave_power_kw(height_m, period_s):
    """Deep-water wave energy flux per metre of wave crest: P = rho g^2 H^2 T / (64 pi) ~ 0.49 H^2 T (kW/m)."""
    return 0.49 * height_m ** 2 * period_s


def _arrow(deg):
    """Arrow pointing where the wind/swell is going (forecast directions say where it comes FROM)."""
    if pd.isna(deg):
        return ""
    return (f"<span title='from {compass(deg)} ({deg:.0f}°)' style='display:inline-block; "
            f"transform:rotate({(deg + 180) % 360:.0f}deg)'>↑</span>")


def _shade(value, steps, colours):
    if pd.isna(value):
        return "#f6f8fa"
    for limit, colour in zip(steps, colours):
        if value < limit:
            return colour
    return colours[-1]


WIND_SHADES = ([10, 20, 30, 40], ["#ffffff", "#e0f2ff", "#9fd8ff", "#5cc8e6", "#ffb86b"])
WAVE_SHADES = ([0.3, 0.6, 1.0, 1.5, 2.5], ["#ffffff", "#ece8ff", "#d6ceff", "#b9adff", "#9b8cff", "#ff9b9b"])


FACTOR_COLUMNS = [("f_size", "Size"), ("f_period", "Period"), ("f_direction", "Direction"),
                  ("f_wind", "Wind"), ("f_tide", "Tide"), ("f_skill", "Level")]


def render_breakdown(info):
    """Transparency table: how every improved result got its condition fit (CF), verdict and final score."""
    results, pq = info["results"], info["parsed"]
    if not results:
        return ""
    t = THRESHOLDS[pq.skill]
    intent = pq.condition_intent
    alpha_note = (f"α = {ALPHA} because the query asks about conditions, time or skill level" if intent else
                  f"α = {ALPHA_NO_CONDITION_INTENT} because the query does not ask about conditions, time or skill level "
                  "(it is treated as an attribute question, so the text counts more)")

    # --- table A: condition fit per spot
    head = ("<tr><th>#</th><th>Spot</th><th>Best hour</th><th>Conditions at that hour</th>"
            + "".join(f"<th>{label}</th>" for _, label in FACTOR_COLUMNS)
            + "<th>CF = product</th><th>Verdict</th></tr>")
    rows_a, rows_b = "", ""
    for r in results:
        f, c, w = r.get("factors") or {}, r.get("conditions") or {}, r.get("weights") or {}
        if not f:
            rows_a += (f"<tr><td>{r['rank']}</td><td>{escape(r['name'])}</td>"
                       f"<td colspan='{len(FACTOR_COLUMNS) + 4}'>No forecast for this spot in the requested time.</td></tr>")
        else:
            limiting = min((k for k, _ in FACTOR_COLUMNS), key=lambda k: f[k])
            cond = (f"{c['wave_height_m']:.1f} m {compass(c['dir_deg'])} {'swell' if c['swell_dominant'] else 'wind swell'}, "
                    f"{c['period_s']:.0f} s · wind {c['wind_kmh']:.0f} km/h {compass(c['wind_dir_deg'])} ({c['wind_label']}) · "
                    f"{c['tide_label']} tide")
            cells, parts = "", []
            for k, _ in FACTOR_COLUMNS:
                weight = w.get(k, 1.0)
                shown = f"{f[k]:.2f}" if weight == 1 else f"{f[k]:.2f}<br><span class='bd-small'>^{weight:g} = {f[k] ** weight:.2f}</span>"
                parts.append(f"{f[k] ** weight:.2f}")
                css = " class='bd-limit'" if k == limiting and r["verdict"] != "Good" else ""
                cells += f"<td{css}>{shown}{'<br><span class=bd-small>▼ limiting</span>' if css else ''}</td>"
            rows_a += (f"<tr><td>{r['rank']}</td><td>{escape(r['name'])}</td><td>{escape(r['best_time'] or '–')}</td>"
                       f"<td class='bd-left'>{escape(cond)}</td>{cells}"
                       f"<td><b>{r['cf']:.2f}</b><br><span class='bd-small'>{' × '.join(parts)}</span></td>"
                       f"<td>{escape(r['verdict'])}</td></tr>")
        # --- table B: final score per spot
        a = r.get("alpha")
        if a is None:
            calc = f"text only = {r['text_score']:.2f}"
        else:
            calc = f"{a:g} × {r['text_score']:.2f} + {1 - a:g} × {r['cf']:.2f}"
            if r.get("safety_penalty"):
                calc += " − 1 (unsafe)"
            if r.get("distance_penalty"):
                calc += f" − {abs(r['distance_penalty']):.3f} (distance)"
        cf_txt = "–" if r["cf"] is None else f"{r['cf']:.2f}"
        alpha_txt = "–" if a is None else f"{a:g}"
        rows_b += (f"<tr><td>{r['rank']}</td><td>{escape(r['name'])}</td><td>{r['text_score']:.2f}</td>"
                   f"<td>{cf_txt}</td><td>{alpha_txt}</td>"
                   f"<td class='bd-left'>{calc}</td><td><b>{r['score']:.2f}</b></td></tr>")

    legend = (f"<div class='meta'>Each factor is between 0 (kills the session) and 1 (perfect). "
              f"CF = size × period × direction × wind × tide<sup>{t['weight_tide']:g}</sup> × level, at the best daylight hour "
              f"of your time window. Verdict for {escape(SKILL_PLURAL[pq.skill])}: Poor below {t['surfable_threshold']:g}, "
              f"Fair from {t['surfable_threshold']:g}, Good from {t['good_threshold']:g}. "
              f"The <span class='bd-limit-key'>red cell</span> is the lowest factor, named in the explanation as "
              f"“Limited by …”. Thresholds for {escape(pq.skill)}: ideal waves {t['wave_ideal_low_m']:g}–{t['wave_ideal_high_m']:g} m "
              f"(unsafe above {t['wave_max_m']:g} m), full period score from {t['period_ideal_s']:g} s, "
              f"onshore wind score 0 from {t['wind_onshore_max_kmh']:g} km/h. All values: data/resources/surfability_thresholds.csv.</div>")
    return ("<h3>How the scores were calculated (improved system)</h3>"
            "<h4>1. Condition fit (CF): the wave conditions</h4>" + legend +
            f"<div class='fc-scroll'><table class='bd'>{head}{rows_a}</table></div>"
            "<h4>2. Final ranking score: text match + conditions</h4>"
            f"<div class='meta'>Score = α × text score + (1 − α) × CF. {escape(alpha_note)}. "
            "The text score (0–1) is the BM25 match of your words with the spot description (1 = best match of all spots). "
            "Spots unsafe for your level get −1 and drop to the bottom.</div>"
            "<div class='fc-scroll'><table class='bd'><tr><th>#</th><th>Spot</th><th>Text score</th><th>CF</th><th>α</th>"
            f"<th>Calculation</th><th>Score</th></tr>{rows_b}</table></div>")


def render_forecast_table(results, snap, windows):
    """Hourly forecast (every 3 h) for the top 3 improved results, like a surf-forecast app."""
    if not results:
        return ""
    ref = snap["ref_time"].floor(f"{FORECAST_STEP_HOURS}h")
    start = ref
    if windows:
        first = min(s for s, _ in windows).floor(f"{FORECAST_STEP_HOURS}h")
        start = max(ref, first)
    times = pd.date_range(start, periods=FORECAST_COLUMNS, freq=f"{FORECAST_STEP_HOURS}h")
    in_window = [any(s <= t < e for s, e in windows) for t in times]
    marine, wind = snap["marine"], snap["wind"]

    day_cells, col = "", 0
    for day, n in pd.Series(times.date).value_counts(sort=False).items():
        day_cells += f"<th colspan='{n}' class='fc-day'>{pd.Timestamp(day):%A %d %b}</th>"
    hour_cells = "".join(f"<th class='{'fc-win' if w else ''}'>{t:%H}</th>" for t, w in zip(times, in_window))

    tables = []
    for r in results[:3]:
        m = marine[marine["spot_id"] == r["spot_id"]].set_index("time").reindex(times)
        w = wind[wind["spot_id"] == r["spot_id"]].set_index("time").reindex(times)
        temp = w["temperature_2m"] if "temperature_2m" in w else pd.Series(np.nan, index=times)
        power = wave_power_kw(m["wave_height"], m["wave_period"])

        def row(label, unit, values, fmt, shade=None, arrows=None):
            cells = ""
            for i, t in enumerate(times):
                val = values.iloc[i]
                txt = "–" if pd.isna(val) else fmt(val)
                arrow = _arrow(arrows.iloc[i]) + " " if arrows is not None and not pd.isna(val) else ""
                bg = _shade(val, *shade) if shade else ("#fffbe6" if in_window[i] else "#ffffff")
                cells += f"<td style='background:{bg}'>{arrow}{txt}</td>"
            return f"<tr><th class='fc-label'>{label}<br><span class='fc-unit'>{unit}</span></th>{cells}</tr>"

        body = "".join([
            row("Temperature", "°C", temp, lambda x: f"{x:.0f}°"),
            row("Wind", "km/h", w["wind_speed_10m"], lambda x: f"{x:.0f}", WIND_SHADES, w["wind_direction_10m"]),
            row("Wind gusts", "km/h", w["wind_gusts_10m"], lambda x: f"{x:.0f}", WIND_SHADES),
            row("Waves", "m", m["wave_height"], lambda x: f"{x:.1f}", WAVE_SHADES, m["wave_direction"]),
            row("Swell", "m", m["swell_wave_height"], lambda x: f"{x:.1f}", WAVE_SHADES, m["swell_wave_direction"]),
            row("Swell period", "s", m["swell_wave_period"], lambda x: f"{x:.1f}"),
            row("Wave power", "kW/m", power, lambda x: f"{x:.0f}"),
        ])
        tables.append(
            f"<div class='fc-spot'><b>{r['rank']}. {escape(r['name'])}</b> "
            f"<span class='meta'>· {escape(r['verdict'])} · best {escape(r['best_time'] or '–')}</span>"
            f"<div class='fc-scroll'><table class='fc'><tr><th></th>{day_cells}</tr>"
            f"<tr><th class='fc-label'>Hour</th>{hour_cells}</tr>{body}</table></div></div>")
    note = ("<div class='meta'>Every 3 hours, Taiwan time. Highlighted hours = the time window of your query. "
            "Arrows point where the wind and swell are going. Wave power ≈ 0.49 × wave height² × wave period "
            "(deep-water wave energy flux). Temperature: “–” in snapshots collected before it was added.</div>")
    return "<h3>Forecast for the top 3 spots (improved system)</h3>" + note + "".join(tables)

## 3. The search callback
Both columns call the same `search_verbose()` used by the evaluation, on the same snapshot and with the same overrides, so what the audience sees is exactly what was evaluated.

**Progress bar.** The first *Live data* search downloads forecasts for every spot (~30 s). Without feedback the page looks frozen, so the callback reports each step to Gradio's progress bar: what is being loaded, how many seconds have passed, and an estimate of the time left (elapsed time ÷ share done × share remaining).

In [20]:
def make_reporter(progress):
    """Returns report(share_done, label) that updates the progress bar with elapsed and estimated remaining time."""
    started = time.perf_counter()

    def report(share, label):
        if progress is None:
            return
        elapsed = time.perf_counter() - started
        left = f" · about {elapsed / share * (1 - share):.0f} s left" if share > 0.02 and elapsed > 1 else ""
        progress(share, desc=f"{label} · {elapsed:.0f} s elapsed{left}")
    return report


def run_search(query, skill_choice, day_choice, source, snapshot_date, k, progress=gr.Progress()):
    if not query or not query.strip():
        return "Please type a query.", "", "", "", "", ""
    report = make_reporter(progress)
    report(0.0, "Starting search")
    live_step = lambda done, total, label: report(0.9 * done / max(total, 1), f"{label} ({100 * done / max(total, 1):.0f}% downloaded)")
    if source == "Live data":
        report(0.0, "Loading live data (first live search takes about 30 s, then cached for 30 min)")
    else:
        report(0.1, "Loading saved snapshot")
    snap, notice = pick_snapshot(source, snapshot_date, progress=live_step)
    ref = snap["ref_time"]
    skill = None if skill_choice == "From query" else skill_choice
    day = None if day_choice == "From query" else (ref.normalize() + pd.Timedelta(days=DAY_CHOICES.index(day_choice) - 1)).date().isoformat()
    report(0.92, "Ranking with the baseline (BM25)")
    base = search_verbose(query, "baseline", int(k), snapshot=snap, skill=skill, day=day)
    report(0.95, "Ranking with the improved system")
    imp = search_verbose(query, "improved", int(k), snapshot=snap, skill=skill, day=day)
    report(0.98, "Building the 7-day outlook")
    pq = imp["parsed"]
    source_line = (f"Live data fetched {ref:%Y-%m-%d %H:%M} (Taiwan time)" if snap["key"].startswith("live")
                   else f"Saved snapshot {snap['date']} (fetched {snap['meta']['fetched_at'].replace('T', ' ')} Taiwan time)")
    info = [f"**Understood:** {describe_parsed(pq)}"]
    if imp["expansions"]:
        info.append(f"**Jargon expansion:** {', '.join(sorted(set(imp['expansions'])))}")
    info += [f"**Notice:** {n}" for n in pq.notes + ([notice] if notice else [])]
    info.append(f"**Data:** {source_line}")
    label_b, label_i = ("Baseline (BM25)", "Improved (condition-aware)") if SHOW_SYSTEM_NAMES else ("System A", "System B")
    head_b = f"<h3>{label_b}</h3><div class='meta'>{base['latency_ms']['total']:.0f} ms</div>"
    head_i = (f"<h3>{label_i}</h3><div class='meta'>{imp['latency_ms']['total']:.0f} ms "
              f"(understanding {imp['latency_ms']['understanding']:.0f} · text {imp['latency_ms']['text']:.0f} · "
              f"conditions {imp['latency_ms']['conditions']:.0f})</div>")
    return ("  \n".join(info), head_b + render_cards(base, show_conditions=False),
            head_i + render_cards(imp, show_conditions=True), render_outlook(imp["results"], snap, pq.skill),
            render_breakdown(imp), render_forecast_table(imp["results"], snap, pq.windows))

## 4. The interface
Required data attribution (DWD / Open-Meteo / CWA) is always visible at the bottom of the page. The **Open surf spot map** button (top right) opens the interactive map from `07_spot_map` in a new tab, and every result card has a **Show on map** link that opens the map zoomed to that spot. If the map file does not exist yet, run `07_spot_map.ipynb` once.

In [21]:
ATTRIBUTION = (
    "**Data sources.** Wave, swell, sea-level and wind forecasts: [Open-Meteo.com](https://open-meteo.com/) "
    "(licensed CC BY 4.0), including wave model data from the Deutscher Wetterdienst (DWD). "
    "Buoy and tide observations: Central Weather Administration (CWA) Open Data, Taiwan. "
    "Spot profiles: written by the project team from cited public sources. "
    "Forecasts describe offshore conditions; always check conditions on the beach. Educational project (MIS590), non-commercial."
)


MAP_BUTTON = (
    f"<div style='text-align:right; padding-top:18px'><a href='{MAP_URL}' target='_blank' "
    "style='display:inline-block; padding:10px 18px; border-radius:8px; background:#1f6feb; color:white; "
    "font-weight:600; text-decoration:none'>🗺️ Open surf spot map</a></div>"
    if MAP_FILE.exists() else
    "<div style='text-align:right; padding-top:18px; color:#57606a'>🗺️ Map not found: run 07_spot_map.ipynb once</div>"
)


def build_demo():
    snapshots = list_snapshots()
    with gr.Blocks(title="Surf Spot Search Taiwan", css=CSS, theme=gr.themes.Soft()) as app:
        with gr.Row():
            gr.Markdown("# 🏄 Surf Spot Search Taiwan\nAsk in plain English, e.g. *beginner-friendly beach break near Kaohsiung this weekend*.")
            gr.HTML(MAP_BUTTON)
        with gr.Row():
            query = gr.Textbox(label="Query", placeholder="uncrowded reef break on the east coast tomorrow morning", scale=4)
            search_btn = gr.Button("Search", variant="primary", scale=1)
        with gr.Row():
            skill = gr.Dropdown(SKILL_CHOICES, value="From query", label="Skill level")
            day = gr.Dropdown(DAY_CHOICES, value="From query", label="Day")
            source = gr.Radio(["Saved snapshot", "Live data"], value="Saved snapshot", label="Data")
            snapshot = gr.Dropdown(snapshots, value=DEMO_SNAPSHOT_DATE or (snapshots[-1] if snapshots else None), label="Snapshot")
            k = gr.Slider(1, 10, value=DEFAULT_K, step=1, label="Results")
        gr.Examples(EXAMPLE_QUERIES, inputs=[query])
        info = gr.Markdown()
        with gr.Row():
            baseline_col = gr.HTML()
            improved_col = gr.HTML()
        outlook = gr.HTML()
        breakdown = gr.HTML()
        forecast_table = gr.HTML()
        gr.Markdown(ATTRIBUTION)
        inputs = [query, skill, day, source, snapshot, k]
        outputs = [info, baseline_col, improved_col, outlook, breakdown, forecast_table]
        search_btn.click(run_search, inputs, outputs, show_progress="full")
        query.submit(run_search, inputs, outputs, show_progress="full")
    return app

## 5. Self-check before launching
Runs one example through the exact callback the interface uses. If this cell fails, the demo would fail too, so you find out before the audience does.

In [22]:
_check = run_search(EXAMPLE_QUERIES[0], "From query", "From query", "Saved snapshot", DEMO_SNAPSHOT_DATE, 3, progress=None)
print(_check[0].replace("**", ""))
print("Baseline and improved columns rendered:", all(len(x) > 0 for x in _check[1:3]))
print("Score breakdown rendered:", "How the scores were calculated" in _check[4])
print("Forecast table rendered:", "Forecast for the top 3 spots" in _check[5])

Understood: Location: Kaohsiung · Skill: beginner · Time: this weekend  
Data: Saved snapshot 2026-10-16 (fetched 2026-10-16 11:21:03 Taiwan time)
Baseline and improved columns rendered: True
Score breakdown rendered: True
Forecast table rendered: True


## 6. Launch
The app appears inline below; the printed URL opens the same app in a browser (use that one for the presentation, full screen with F11). Re-running this cell restarts the server.

In [23]:
if "demo" in globals():
    try:
        demo.close()
    except Exception:
        pass
demo = build_demo()
demo.launch(server_name=SERVER_NAME, server_port=SERVER_PORT, share=SHARE, inline=True,
            prevent_thread_lock=True, quiet=True, allowed_paths=[str(MAP_FILE.parent)])
print(f"Demo running at {demo.local_url}  (open in a browser for full screen)")

Closing server running on port: 7860


Demo running at http://127.0.0.1:7860/  (open in a browser for full screen)


In [24]:
STOP_DEMO = False      # set to True and run this cell to stop the server
if STOP_DEMO:
    demo.close()